# AI Agent 工程化实战：案例驱动 Notebook

本 Notebook 以 [AI_Agent工程化实战_合并版.pptx](../docs/AI_Agent工程化实战_合并版.pptx) 为背景，但不重复讲 PPT 内容，重点放在 3 类可运行案例：

1. 本地课程资料检索与工具调用
2. 基于真实语料的 Agent 问答与反馈循环
3. 基于外部实时数据的 Agent 简报

模型初始化方式对齐 `prompt_engineering_complete.ipynb`，真实数据来自课程仓库、PPT 文本和实时 API。

## Notebook 路径

1. 加载模型与环境
2. 提取 PPT 标题，确认案例范围
3. 构建本地检索与工具层
4. 运行离线工具烟雾测试
5. 运行课程资料 Agent 案例
6. 运行实时数据 Agent 案例
7. 查看护栏评估结果

In [59]:
import ast
import json
import os
import re
import sqlite3
import sys
import time
from dataclasses import asdict, dataclass, field
from enum import Enum
from pathlib import Path
from textwrap import dedent
from typing import Any, Callable, Dict, Iterable, List, Optional
from zipfile import ZipFile
from xml.etree import ElementTree as ET

import requests
from dotenv import load_dotenv
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

PROJECT_ROOT = Path.cwd().parent
REPO_ROOT = PROJECT_ROOT.parent
PROMPT_ROOT = REPO_ROOT / 'prompt-engineering-course'
sys.path.insert(0, str(PROMPT_ROOT))

from config import LLMConfig
from utils.llm_client import LLMClient

# 对齐 prompt_engineering_complete.ipynb：显式从 prompt-engineering-course/.env 加载配置
load_dotenv(dotenv_path=PROMPT_ROOT / '.env')

QWEN_API_KEY = os.environ.get('QWEN_API_KEY')
QWEN_BASE_URL = os.environ.get('QWEN_BASE_URL')

PPT_PATH = PROJECT_ROOT / 'docs' / 'AI_Agent工程化实战_合并版.pptx'
COURSE_README = PROJECT_ROOT / 'README.md'
COURSE_OUTLINE = PROJECT_ROOT / 'docs' / 'AI_Harness_Engineering_课程大纲.md'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('PPT exists =', PPT_PATH.exists())
print('QWEN_API_KEY loaded =', bool(QWEN_API_KEY))
print('QWEN_BASE_URL loaded =', bool(QWEN_BASE_URL))


PROJECT_ROOT = /Users/angusfan/Downloads/三门课程完整版/harness-engineering-course
PPT exists = True
QWEN_API_KEY loaded = True
QWEN_BASE_URL loaded = True


In [60]:
# 参考 prompt_engineering_complete.ipynb 的初始化方式
config = LLMConfig(provider='qwen', model='ClawClaw', temperature=0.3, max_tokens=8192)
client = LLMClient(config)


def llm_is_ready() -> bool:
    return bool(QWEN_API_KEY and QWEN_BASE_URL and client.config.api_key and client.config.base_url)


def llm_chat(messages: List[Dict[str, str]], temperature: Optional[float] = None, **kwargs) -> str:
    if not llm_is_ready():
        raise RuntimeError(
            '未检测到可用的 Qwen 配置。请检查 prompt-engineering-course/.env 中的 QWEN_API_KEY 和 QWEN_BASE_URL。'
        )
    return client.chat(messages, temperature=temperature, **kwargs)


print('provider =', client.config.provider)
print('model =', client.config.model)
print('client api_key loaded =', bool(client.config.api_key))
print('client base_url loaded =', bool(client.config.base_url))
print('llm ready =', llm_is_ready())


provider = qwen
model = ClawClaw
client api_key loaded = True
client base_url loaded = True
llm ready = True


## 1. 用 PPT 只做“案例范围确认”

这里只提取标题和章节，不重复展开讲义内容。

In [61]:
PPT_NS = {
    'a': 'http://schemas.openxmlformats.org/drawingml/2006/main',
    'p': 'http://schemas.openxmlformats.org/presentationml/2006/main',
}


def extract_ppt_texts(ppt_path: Path) -> List[Dict[str, Any]]:
    slides: List[Dict[str, Any]] = []
    with ZipFile(ppt_path) as zf:
        slide_names = sorted(
            [name for name in zf.namelist() if name.startswith('ppt/slides/slide') and name.endswith('.xml')],
            key=lambda x: int(re.search(r'slide(\d+)\.xml', x).group(1)),
        )
        for name in slide_names:
            slide_no = int(re.search(r'slide(\d+)\.xml', name).group(1))
            root = ET.fromstring(zf.read(name))
            texts = [node.text.strip() for node in root.findall('.//a:t', PPT_NS) if node.text and node.text.strip()]
            slides.append({
                'slide_no': slide_no,
                'title': texts[0] if texts else f'Slide {slide_no}',
                'texts': texts,
            })
    return slides


slides = extract_ppt_texts(PPT_PATH)
print('slides =', len(slides))
for slide in slides[:8]:
    print(f"{slide['slide_no']:02d}. {slide['title']}")


slides = 28
01. AI Agent 工程化实战
02. 课程目录
03. 什么是 Harness Engineering？
04. 为什么需要 Harness Engineering？
05. AI 工程范式的三次跃迁
06. Agent 三大失败模式
07. 驾驭工程四大护栏
08. PART 02


In [62]:
chapter_slides = [1, 2, 8, 12, 21, 27, 28]
for slide_id in chapter_slides:
    slide = slides[slide_id - 1]
    print(f"Slide {slide_id:02d}: {slide['title']}")


Slide 01: AI Agent 工程化实战
Slide 02: 课程目录
Slide 08: PART 02
Slide 12: PART 03
Slide 21: PART 05
Slide 27: 核心要点回顾
Slide 28: 动手实践吧！


## 2. 案例底座：本地知识库检索

先把课程 README、课程大纲和示例代码切块建索引。后续两个 Agent 案例都会复用这一层。

In [63]:
@dataclass
class Chunk:
    chunk_id: str
    path: str
    title: str
    content: str


class CourseRetriever:
    def __init__(self) -> None:
        self.conn: Optional[sqlite3.Connection] = None
        self.chunks: List[Chunk] = []
        self.fts_ready = False
        try:
            self.conn = sqlite3.connect(':memory:')
            self.conn.execute('CREATE VIRTUAL TABLE chunks USING fts5(chunk_id, path, title, content)')
            self.fts_ready = True
        except sqlite3.OperationalError:
            self.conn = None
            self.fts_ready = False

    def iter_source_files(self) -> Iterable[Path]:
        candidates = [COURSE_README, COURSE_OUTLINE]
        candidates.extend(sorted((PROJECT_ROOT / 'examples').glob('*.py')))
        return candidates

    def split_text(self, text: str, max_chars: int = 700) -> List[str]:
        text = text.replace('\u3000', ' ').strip()
        blocks = [block.strip() for block in re.split(r'\n\s*\n', text) if block.strip()]
        chunks: List[str] = []
        current = ''
        for block in blocks:
            candidate = block if not current else current + '\n\n' + block
            if len(candidate) <= max_chars:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                if len(block) <= max_chars:
                    current = block
                else:
                    for i in range(0, len(block), max_chars):
                        chunks.append(block[i:i + max_chars])
                    current = ''
        if current:
            chunks.append(current)
        return chunks

    def build(self) -> None:
        self.chunks.clear()
        if self.conn is not None:
            self.conn.execute('DELETE FROM chunks')

        for path in self.iter_source_files():
            text = path.read_text(encoding='utf-8')
            parts = self.split_text(text)
            for idx, part in enumerate(parts, start=1):
                chunk = Chunk(
                    chunk_id=f'{path.name}-{idx}',
                    path=str(path.relative_to(REPO_ROOT)),
                    title=path.name,
                    content=part,
                )
                self.chunks.append(chunk)
                if self.conn is not None:
                    self.conn.execute(
                        'INSERT INTO chunks(chunk_id, path, title, content) VALUES (?, ?, ?, ?)',
                        (chunk.chunk_id, chunk.path, chunk.title, chunk.content),
                    )
        if self.conn is not None:
            self.conn.commit()

    def _fts_search(self, query: str, limit: int) -> List[Dict[str, Any]]:
        if not self.fts_ready or self.conn is None:
            return []
        try:
            sql = dedent('''
                SELECT chunk_id, path, title, snippet(chunks, 3, '[', ']', '...', 10) AS snippet, bm25(chunks) AS score
                FROM chunks
                WHERE chunks MATCH ?
                ORDER BY score
                LIMIT ?
            ''')
            rows = self.conn.execute(sql, (query, limit)).fetchall()
            return [
                {
                    'chunk_id': row[0],
                    'path': row[1],
                    'title': row[2],
                    'snippet': row[3],
                    'score': float(row[4]),
                    'method': 'fts5',
                }
                for row in rows
            ]
        except sqlite3.OperationalError:
            return []

    def _fallback_search(self, query: str, limit: int) -> List[Dict[str, Any]]:
        tokens = [tok for tok in re.split(r'\s+', query.strip()) if tok]
        if not tokens and query.strip():
            tokens = [query.strip()]
        scored = []
        for chunk in self.chunks:
            score = 0
            for tok in tokens:
                score += chunk.content.count(tok) * 10
                if tok in chunk.path:
                    score += 5
            if score > 0:
                scored.append((score, chunk))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [
            {
                'chunk_id': chunk.chunk_id,
                'path': chunk.path,
                'title': chunk.title,
                'snippet': chunk.content[:240].replace('\n', ' '),
                'score': score,
                'method': 'fallback',
            }
            for score, chunk in scored[:limit]
        ]

    def search(self, query: str, limit: int = 5) -> List[Dict[str, Any]]:
        query = query.strip()
        if not query:
            return []
        results = self._fts_search(query, limit)
        if len(results) >= max(2, min(limit, 3)):
            return results
        seen = {item['chunk_id'] for item in results}
        extra = [item for item in self._fallback_search(query, limit) if item['chunk_id'] not in seen]
        return (results + extra)[:limit]


retriever = CourseRetriever()
retriever.build()
print('chunks =', len(retriever.chunks), 'fts5 =', retriever.fts_ready)


chunks = 263 fts5 = True


In [64]:
for query in ['反馈循环', 'Plan-and-Execute']:
    print(f'\n=== {query} ===')
    for item in retriever.search(query, limit=2):
        print(f"- [{item['method']}] {item['path']} :: {item['snippet'][:120]}")



=== 反馈循环 ===
- [fts5] harness-engineering-course/README.md ::                            │
│                   ┌──────────────┐                             │
│                   │   

=== Plan-and-Execute ===
- [fallback] harness-engineering-course/README.md :: **ReAct 的关键优势：** - 推理过程透明可见 - 能够处理多步骤复杂任务 - 可以根据观察结果调整后续行动  ### 2.2 Plan-and-Execute 模式  Plan-and-Execute（计划-执行）模式将任务处理分
- [fallback] harness-engineering-course/README.md :: # 运行     async def main():         result = await agent.run("帮我计算一下 2+3*4 等于多少")         print("=== ReAct执行结果 ===")     


## 3. 案例底座：可控工具层

这里注册 5 个真实工具：课程检索、文件读取、Python 结构解析、安全计算、实时新闻抓取。后续案例直接通过这些工具执行。

In [65]:
@dataclass
class ToolSpec:
    name: str
    description: str
    parameters: Dict[str, Any]
    func: Callable[..., Any]


class ToolRegistry:
    def __init__(self) -> None:
        self._tools: Dict[str, ToolSpec] = {}

    def register(self, tool: ToolSpec) -> None:
        self._tools[tool.name] = tool

    def get(self, name: str) -> ToolSpec:
        return self._tools[name]

    def list(self) -> List[ToolSpec]:
        return [self._tools[name] for name in sorted(self._tools)]

    def schema_text(self) -> str:
        lines: List[str] = []
        for tool in self.list():
            params = json.dumps(tool.parameters, ensure_ascii=False, indent=2)
            lines.append(
                f"工具名: {tool.name}\n描述: {tool.description}\n参数:```json\n{params}\n```"
            )
        return '\n\n'.join(lines)


REPO_ROOT_RESOLVED = REPO_ROOT.resolve()


def ensure_repo_relative(relative_path: str) -> Path:
    target = (REPO_ROOT / relative_path).resolve()
    if target != REPO_ROOT_RESOLVED and REPO_ROOT_RESOLVED not in target.parents:
        raise ValueError('路径越权：只能访问当前课程仓库内的文件')
    if not target.exists():
        raise FileNotFoundError(relative_path)
    return target


def search_course_materials(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """检索课程资料，返回最相关的文本片段。"""
    return retriever.search(query, limit=top_k)


def read_course_file(relative_path: str, max_chars: int = 1800) -> str:
    """读取课程仓库中的 UTF-8 文本文件。"""
    path = ensure_repo_relative(relative_path)
    if path.suffix.lower() not in {'.md', '.py', '.txt', '.json', '.ipynb'}:
        raise ValueError('为了安全起见，这个示例只允许读取文本类文件')
    text = path.read_text(encoding='utf-8')
    return text[:max_chars]


def extract_python_symbols(relative_path: str) -> Dict[str, List[str]]:
    """解析 Python 文件中的函数、类与导入。"""
    path = ensure_repo_relative(relative_path)
    tree = ast.parse(path.read_text(encoding='utf-8'))
    funcs, classes, imports = [], [], []
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            funcs.append(node.name)
        elif isinstance(node, ast.ClassDef):
            classes.append(node.name)
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            imports.append(ast.unparse(node))
    return {
        'functions': sorted(set(funcs)),
        'classes': sorted(set(classes)),
        'imports': imports[:20],
    }


def safe_calculator(expression: str) -> float:
    """安全执行四则运算与括号表达式。"""
    allowed_nodes = (
        ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
        ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.Mod, ast.USub, ast.UAdd,
        ast.Load, ast.FloorDiv,
    )
    tree = ast.parse(expression, mode='eval')
    for node in ast.walk(tree):
        if not isinstance(node, allowed_nodes):
            raise ValueError(f'不允许的表达式节点: {type(node).__name__}')
    return eval(compile(tree, '<safe_calculator>', 'eval'), {'__builtins__': {}}, {})


def fetch_hn_top_stories(limit: int = 10) -> List[Dict[str, Any]]:
    """获取 Hacker News Top Stories。联网时返回实时数据。"""
    if limit < 1 or limit > 30:
        raise ValueError('limit 必须在 1..30 之间')
    ids = requests.get('https://hacker-news.firebaseio.com/v0/topstories.json', timeout=15).json()[:limit]
    stories = []
    for story_id in ids:
        item = requests.get(f'https://hacker-news.firebaseio.com/v0/item/{story_id}.json', timeout=15).json()
        stories.append({
            'id': item.get('id'),
            'title': item.get('title'),
            'url': item.get('url'),
            'score': item.get('score'),
            'by': item.get('by'),
        })
    return stories


tool_registry = ToolRegistry()
tool_registry.register(ToolSpec(
    name='search_course_materials',
    description='检索课程资料，回答课程概念、架构、护栏、工具协议等问题。',
    parameters={
        'type': 'object',
        'properties': {
            'query': {'type': 'string'},
            'top_k': {'type': 'integer', 'default': 5},
        },
        'required': ['query'],
    },
    func=search_course_materials,
))
tool_registry.register(ToolSpec(
    name='read_course_file',
    description='读取课程仓库中的文本文件。',
    parameters={
        'type': 'object',
        'properties': {
            'relative_path': {'type': 'string'},
            'max_chars': {'type': 'integer', 'default': 1800},
        },
        'required': ['relative_path'],
    },
    func=read_course_file,
))
tool_registry.register(ToolSpec(
    name='extract_python_symbols',
    description='解析 Python 文件中的函数、类和导入。',
    parameters={
        'type': 'object',
        'properties': {'relative_path': {'type': 'string'}},
        'required': ['relative_path'],
    },
    func=extract_python_symbols,
))
tool_registry.register(ToolSpec(
    name='safe_calculator',
    description='安全执行数学表达式。',
    parameters={
        'type': 'object',
        'properties': {'expression': {'type': 'string'}},
        'required': ['expression'],
    },
    func=safe_calculator,
))
tool_registry.register(ToolSpec(
    name='fetch_hn_top_stories',
    description='抓取 Hacker News 实时热门新闻，适合演示外部实时数据工具。',
    parameters={
        'type': 'object',
        'properties': {'limit': {'type': 'integer', 'default': 10}},
        'required': [],
    },
    func=fetch_hn_top_stories,
))

print('registered tools =', [tool.name for tool in tool_registry.list()])


registered tools = ['extract_python_symbols', 'fetch_hn_top_stories', 'read_course_file', 'safe_calculator', 'search_course_materials']


In [66]:
print('--- tool_registry_summary ---')
for tool in tool_registry.list():
    print(f"- {tool.name}: {tool.description}")


--- tool_registry_summary ---
- extract_python_symbols: 解析 Python 文件中的函数、类和导入。
- fetch_hn_top_stories: 抓取 Hacker News 实时热门新闻，适合演示外部实时数据工具。
- read_course_file: 读取课程仓库中的文本文件。
- safe_calculator: 安全执行数学表达式。
- search_course_materials: 检索课程资料，回答课程概念、架构、护栏、工具协议等问题。


## 4. 案例底座：Harness Runtime

运行时只保留案例需要的能力：输入护栏、工具参数校验、trace、短期记忆。

In [67]:
class TraceStatus(Enum):
    OK = 'ok'
    BLOCKED = 'blocked'
    ERROR = 'error'


@dataclass
class TraceEvent:
    name: str
    status: TraceStatus
    detail: Dict[str, Any] = field(default_factory=dict)
    timestamp: float = field(default_factory=time.time)


@dataclass
class MemoryItem:
    role: str
    content: str


class ConversationMemory:
    def __init__(self, max_items: int = 12) -> None:
        self.max_items = max_items
        self.items: List[MemoryItem] = []

    def add(self, role: str, content: str) -> None:
        self.items.append(MemoryItem(role=role, content=content))
        if len(self.items) > self.max_items:
            self.items = self.items[-self.max_items:]

    def render(self) -> str:
        return '\n'.join(f'[{item.role}] {item.content}' for item in self.items)


class HarnessRuntime:
    blocked_patterns = [
        r'删除.*数据库',
        r'drop\s+table',
        r'泄露.*密钥',
        r'api[_-]?key',
        r'读取\s+/.+',
    ]

    def __init__(self, registry: ToolRegistry) -> None:
        self.registry = registry
        self.trace: List[TraceEvent] = []
        self.memory = ConversationMemory()

    def record(self, name: str, status: TraceStatus, **detail: Any) -> None:
        self.trace.append(TraceEvent(name=name, status=status, detail=detail))

    def validate_user_goal(self, user_goal: str) -> None:
        for pattern in self.blocked_patterns:
            if re.search(pattern, user_goal, flags=re.IGNORECASE):
                raise PermissionError(f'输入命中安全规则: {pattern}')

    def validate_tool_call(self, tool_name: str, args: Dict[str, Any]) -> None:
        if tool_name in {'read_course_file', 'extract_python_symbols'}:
            path = str(args.get('relative_path', ''))
            if '..' in path or path.startswith('/'):
                raise PermissionError('工具参数越权：只允许访问课程仓库内的相对路径')
        if tool_name == 'fetch_hn_top_stories' and int(args.get('limit', 10)) > 30:
            raise PermissionError('limit 过大，已拒绝执行')

    def execute_tool(self, tool_name: str, args: Dict[str, Any]) -> Any:
        self.validate_tool_call(tool_name, args)
        tool = self.registry.get(tool_name)
        started_at = time.time()
        try:
            result = tool.func(**args)
            self.record('tool_call', TraceStatus.OK, tool=tool_name, args=args, elapsed=round(time.time() - started_at, 4))
            return result
        except Exception as exc:
            self.record('tool_call', TraceStatus.ERROR, tool=tool_name, args=args, error=str(exc))
            raise


runtime = HarnessRuntime(tool_registry)
print('runtime ready')


runtime ready


## 5. 案例底座：JSON Action Agent

这个 Agent 不讲框架概念，直接演示最小闭环：

- LLM 决定下一步
- 工具真实执行
- 结果回写上下文
- 最终答案带引用
- 验证器检查 grounding

In [68]:
def extract_json_object(text: str) -> Dict[str, Any]:
    text = text.strip()
    if text.startswith('```json'):
        text = text.split('```json', 1)[1].split('```', 1)[0].strip()
    elif text.startswith('```'):
        text = text.split('```', 1)[1].split('```', 1)[0].strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


AGENT_SYSTEM_PROMPT = dedent('''
你是一个面向课程演示的 AI Agent，必须遵守以下规则：

1. 你只能输出 JSON，不允许输出解释性前言。
2. 如果需要外部信息，优先调用工具，而不是臆测。
3. 当问题涉及课程内容时，优先使用 `search_course_materials` 检索，再按需读取文件。
4. 如果你要结束任务，必须输出 `action = "final"`，并给出 `citations`。
5. citations 必须是你真正看过的资料路径，例如 `harness-engineering-course/README.md`。
6. 遇到工具错误时，先调整策略，不要直接宣称完成。

你可以使用以下工具：
{tool_schemas}

输出 JSON 只能是以下两种格式之一：

工具调用：
{{
  "action": "tool",
  "thought": "一句简短思考",
  "tool_name": "工具名",
  "args": {{"参数": "值"}}
}}

最终回答：
{{
  "action": "final",
  "thought": "一句简短总结",
  "answer": "最终答案，使用中文",
  "citations": ["路径1", "路径2"]
}}
''')


class JSONActionAgent:
    def __init__(self, runtime: HarnessRuntime, max_iterations: int = 8) -> None:
        self.runtime = runtime
        self.max_iterations = max_iterations

    def build_messages(self, user_goal: str) -> List[Dict[str, str]]:
        return [
            {
                'role': 'system',
                'content': AGENT_SYSTEM_PROMPT.format(tool_schemas=self.runtime.registry.schema_text()),
            },
            {
                'role': 'user',
                'content': user_goal,
            },
        ]

    def run(self, user_goal: str) -> Dict[str, Any]:
        self.runtime.trace = []
        self.runtime.memory = ConversationMemory()
        self.runtime.validate_user_goal(user_goal)

        messages = self.build_messages(user_goal)
        self.runtime.memory.add('user', user_goal)
        observations: List[Dict[str, Any]] = []

        for iteration in range(1, self.max_iterations + 1):
            raw = llm_chat(messages, temperature=0.1)
            self.runtime.record('llm_response', TraceStatus.OK, iteration=iteration, raw=raw[:500])
            self.runtime.memory.add('assistant', raw)

            try:
                action = extract_json_object(raw)
            except Exception as exc:
                correction = f'你的上一条输出不是合法 JSON。错误: {exc}。请严格按指定 JSON 重新输出。'
                messages.append({'role': 'assistant', 'content': raw})
                messages.append({'role': 'user', 'content': correction})
                self.runtime.record('json_parse', TraceStatus.ERROR, iteration=iteration, error=str(exc))
                continue

            if action.get('action') == 'tool':
                tool_name = action['tool_name']
                args = action.get('args', {})
                result = self.runtime.execute_tool(tool_name, args)
                observations.append({'tool': tool_name, 'args': args, 'result': result})
                tool_payload = json.dumps(result, ensure_ascii=False) if not isinstance(result, str) else result
                messages.append({'role': 'assistant', 'content': raw})
                messages.append({'role': 'user', 'content': f'工具 `{tool_name}` 返回：\n{tool_payload}'})
                self.runtime.memory.add('system', f'{tool_name} -> {tool_payload[:400]}')
                continue

            if action.get('action') == 'final':
                answer = action.get('answer', '').strip()
                citations = action.get('citations', [])
                return {
                    'ok': True,
                    'answer': answer,
                    'citations': citations,
                    'observations': observations,
                    'trace': self.runtime.trace,
                    'memory': self.runtime.memory.render(),
                }

            messages.append({'role': 'assistant', 'content': raw})
            messages.append({'role': 'user', 'content': 'action 字段只能是 tool 或 final。请重试。'})

        return {
            'ok': False,
            'error': 'Max iterations exceeded',
            'observations': observations,
            'trace': self.runtime.trace,
            'memory': self.runtime.memory.render(),
        }


agent = JSONActionAgent(runtime)
print('agent ready')


agent ready


## 6. 案例底座：Feedback Loop

这一层只负责验证答案是否真的引用了执行过程中看过的材料。

In [69]:
@dataclass
class ValidationResult:
    passed: bool
    errors: List[str] = field(default_factory=list)


class GroundingValidator:
    def validate(self, result: Dict[str, Any]) -> ValidationResult:
        if not result.get('ok'):
            return ValidationResult(False, ['agent 未成功完成任务'])

        citations = result.get('citations') or []
        if not citations:
            return ValidationResult(False, ['最终答案缺少 citations'])

        observed_paths = set()
        for obs in result.get('observations', []):
            tool = obs.get('tool')
            data = obs.get('result')
            if tool == 'search_course_materials' and isinstance(data, list):
                observed_paths.update(item.get('path') for item in data if item.get('path'))
            elif tool in {'read_course_file', 'extract_python_symbols'}:
                rel = obs.get('args', {}).get('relative_path')
                if rel:
                    observed_paths.add(rel)
            elif tool == 'fetch_hn_top_stories':
                observed_paths.add('live:HackerNews')

        missing = [cite for cite in citations if cite not in observed_paths]
        if missing:
            return ValidationResult(False, [f'以下引用没有出现在工具观察结果中: {missing}'])

        return ValidationResult(True, [])


class FeedbackLoop:
    def __init__(self, agent: JSONActionAgent, validator: GroundingValidator, max_attempts: int = 2) -> None:
        self.agent = agent
        self.validator = validator
        self.max_attempts = max_attempts

    def refine_goal(self, original_goal: str, errors: List[str]) -> str:
        joined = '; '.join(errors)
        return original_goal + f"\n\n修正要求：上一轮未通过验证。请确保引用真实、答案可追溯。问题：{joined}"

    def run(self, user_goal: str) -> Dict[str, Any]:
        current_goal = user_goal
        result: Dict[str, Any] = {}
        for attempt in range(1, self.max_attempts + 1):
            result = self.agent.run(current_goal)
            validation = self.validator.validate(result)
            result['validation'] = asdict(validation)
            result['attempt'] = attempt
            if validation.passed:
                return result
            current_goal = self.refine_goal(user_goal, validation.errors)
        return result


feedback_loop = FeedbackLoop(agent, GroundingValidator())
print('feedback loop ready')


feedback loop ready


## 案例 0：离线工具烟雾测试

先验证工具层本身可用，再进入 Agent 案例。

In [70]:
print('--- search_course_materials ---')
print(json.dumps(search_course_materials('反馈循环', top_k=2), ensure_ascii=False, indent=2)[:1000])

print('\n--- read_course_file ---')
print(read_course_file('harness-engineering-course/README.md', max_chars=300))

print('\n--- extract_python_symbols ---')
print(json.dumps(extract_python_symbols('harness-engineering-course/examples/06_harness_runtime.py'), ensure_ascii=False, indent=2))

print('\n--- safe_calculator ---')
print(safe_calculator('(2 + 3) * 4 / 5'))

if llm_is_ready():
    print('\n--- direct_llm_baseline ---')
    baseline = llm_chat([
        {'role': 'system', 'content': '你是一名 AI Agent 工程化课程助教。'},
        {'role': 'user', 'content': '请用三句话概括 AI Agent 里的 Harness Engineering，明确指出这是 AI/LLM 场景，不是线束工程。'}
    ], max_tokens=300)
    print(baseline)


--- search_course_materials ---
[
  {
    "chunk_id": "README.md-3",
    "path": "harness-engineering-course/README.md",
    "title": "README.md",
    "snippet": "                           │\n│                   ┌──────────────┐                             │\n│                   │   [反馈循环]    │                             │\n│                   │  自我反思改进 │                             │\n│                   └──────────────┘                             │\n└─────────────────────────────────────────────────────────────────┘\n```",
    "score": -7.847305703337818,
    "method": "fts5"
  }
]

--- read_course_file ---
# 驾驭工程（Harness Engineering）课程

> 面向软件学院本科生/研究生的技术教程

---

## 目录

1. [核心概念定义](#1-核心概念定义)
2. [AI Harness架构与实现](#2-ai-agent架构与实现)
3. [多步骤任务编排与工具调用](#3-多步骤任务编排与工具调用)
4. [输出质量控制与验证](#4-输出质量控制与验证)
5. [实际代码案例](#5-实际代码案例)
6. [适合教学的场景化案例](#6-适合教学的场景化案例)

---

## 1. 核心概念定义

### 1.1 什么是驾驭工程

**驾驭工程（Harness Eng

--- extract_python_symbols ---
{
  "functions": [
    "__init__",
    "answer"

## 案例 1：受控课程资料 Agent

目标：不用放任模型自由循环，而是先明确执行检索与读取，再让模型基于真实观察结果输出答案。这更接近生产里的 Harness 思路。

In [71]:
course_query = 'Harness Engineering、四大护栏、ReAct、Plan-and-Execute'
course_search = runtime.execute_tool('search_course_materials', {'query': course_query, 'top_k': 4})
course_readme = runtime.execute_tool('read_course_file', {'relative_path': 'harness-engineering-course/README.md', 'max_chars': 2200})
course_outline = runtime.execute_tool('read_course_file', {'relative_path': 'harness-engineering-course/docs/AI_Harness_Engineering_课程大纲.md', 'max_chars': 1800})

course_context = {
    'search_results': course_search,
    'readme_excerpt': course_readme,
    'outline_excerpt': course_outline,
}

course_prompt = dedent('''
你是 AI Agent 工程化课程助教，这里的 Harness Engineering 指 AI/LLM 系统的执行外壳与反馈循环，不是传统线束工程。

请基于提供的观察结果，输出一个短答案：
1. 用 2-3 句话解释 Harness Engineering 为什么不只是优化 prompt。
2. 用 4 个要点总结四大护栏，每个要点写明它主要防什么风险。
3. 用 1 句话比较 ReAct 和 Plan-and-Execute 的适用差异。

要求：
- 使用中文
- 控制在 350 字以内
- 只根据观察结果作答，不要编造新来源
''').strip()

course_answer = llm_chat([
    {'role': 'system', 'content': '你必须基于给定观察结果回答，不能把 Harness Engineering 解释为传统线束工程。'},
    {'role': 'user', 'content': f'观察结果如下：\n{json.dumps(course_context, ensure_ascii=False)}\n\n任务：{course_prompt}'},
], max_tokens=900)

print('--- course_case_meta ---')
print(json.dumps({
    'citations': [
        'harness-engineering-course/README.md',
        'harness-engineering-course/docs/AI_Harness_Engineering_课程大纲.md',
    ],
    'search_hits': len(course_search),
    'readme_chars': len(course_readme),
    'outline_chars': len(course_outline),
}, ensure_ascii=False, indent=2))

print('\n--- course_case_search_preview ---')
print(json.dumps(course_search[:2], ensure_ascii=False, indent=2))

display(Markdown('### 案例 1 最终答案'))
display(Markdown(course_answer))


--- course_case_meta ---
{
  "citations": [
    "harness-engineering-course/README.md",
    "harness-engineering-course/docs/AI_Harness_Engineering_课程大纲.md"
  ],
  "search_hits": 4,
  "readme_chars": 2200,
  "outline_chars": 1800
}

--- course_case_search_preview ---
[
  {
    "chunk_id": "AI_Harness_Engineering_课程大纲.md-2",
    "path": "harness-engineering-course/docs/AI_Harness_Engineering_课程大纲.md",
    "title": "AI_Harness_Engineering_课程大纲.md",
    "snippet": "关键转变：从“被动回答”到“目标驱动执行”。严格说，Agent 并不是完全自主的软件生命体，而是在 Harness 约束下反复进行“观察状态、选择动作、执行工具、读取反馈”的工程系统。  #### 知识点 2：什么是 Harness Engineering  **Harness（驾驭系统）** 是包裹在 LLM 核心推理能力之外的一整套工程基础设施，包括：  - **工具注册与调度**（Tool Registry & Dispatch） - **技能/能力扩展**（Skill",
    "score": 55,
    "method": "fallback"
  },
  {
    "chunk_id": "AI_Harness_Engineering_课程大纲.md-1",
    "path": "harness-engineering-course/docs/AI_Harness_Engineering_课程大纲.md",
    "title": "AI_Harness_Engineering_课程大纲.md",
    "snippet": "# AI Harness Engineering 实训课程大纲  > **授课对象**：软件

### 案例 1 最终答案

Harness Engineering 是包裹在 LLM 核心推理之外的完整工程基础设施，包含工具调度、记忆管理及安全控制，而非仅优化提示词。它通过构建“观察 - 行动 - 反馈”的闭环系统，将被动回答转化为受约束的目标驱动执行，确保 AI 在真实环境中稳定运行。

四大护栏及其防御风险如下：
1. **工具注册与调度**：防止 Agent 调用未授权或危险的外部 API 及函数。
2. **安全与权限控制**：防御越权操作、敏感数据泄露及恶意指令执行。
3. **输出质量控制与验证**：规避模型幻觉、逻辑错误及不符合业务规范的输出。
4. **会话生命周期管理**：避免状态丢失、上下文溢出及无限循环导致的系统崩溃。

ReAct 适用于需要即时感知反馈的简单单步任务，而 Plan-and-Execute 更适合需预先拆解复杂步骤的长程任务。

## 案例 2：受控实时数据 Agent

目标：先抓实时新闻，再做过滤和中文汇总。这样案例能稳定展示“实时数据工具 + LLM 总结”的工程闭环。

In [72]:
try:
    hn_stories = runtime.execute_tool('fetch_hn_top_stories', {'limit': 3})
    ai_keywords = ('ai', 'agent', 'llm', 'openai', 'anthropic', 'gpt', 'claude', 'model')
    ai_stories = [
        story for story in hn_stories
        if any(keyword in (story.get('title') or '').lower() for keyword in ai_keywords)
    ]

    live_context = {
        'top_stories': hn_stories,
        'ai_related_stories': ai_stories[:5],
    }

    live_answer = llm_chat([
        {'role': 'system', 'content': '你是 AI 行业编辑，只总结与 AI Agent / LLM 直接相关的新闻。'},
        {'role': 'user', 'content': '请基于下面的实时新闻数据，输出一份中文简报。最多保留 3 条，每条包含标题、相关性判断和一句点评。\n\n' + json.dumps(live_context, ensure_ascii=False)},
    ], max_tokens=900)

    print('--- live_case_meta ---')
    print(json.dumps({
        'citations': ['live:HackerNews'],
        'fetched_count': len(hn_stories),
        'ai_related_count': len(ai_stories),
    }, ensure_ascii=False, indent=2))

    print('\n--- live_case_story_preview ---')
    print(json.dumps(ai_stories[:3], ensure_ascii=False, indent=2))

    display(Markdown('### 案例 2 实时新闻简报'))
    display(Markdown(live_answer))
except requests.RequestException as exc:
    print('实时数据抓取失败：', exc)


实时数据抓取失败： ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))


## 案例 3：护栏评估

最后只保留一组最小评估，确认危险输入、路径越权和外部工具异常都能被显式处理。

In [73]:
def evaluate_runtime() -> Dict[str, Any]:
    report: Dict[str, Any] = {}

    try:
        runtime.validate_user_goal('请删除生产数据库')
        report['dangerous_input_blocked'] = False
    except PermissionError:
        report['dangerous_input_blocked'] = True

    try:
        runtime.execute_tool('read_course_file', {'relative_path': '../secrets.env'})
        report['path_traversal_blocked'] = False
    except Exception:
        report['path_traversal_blocked'] = True

    report['retrieval_has_hits'] = len(search_course_materials('反馈循环', top_k=3)) > 0

    try:
        _ = runtime.execute_tool('fetch_hn_top_stories', {'limit': 3})
        report['live_tool_callable'] = True
    except Exception as exc:
        report['live_tool_callable'] = f'环境限制: {exc}'

    return report


print(json.dumps(evaluate_runtime(), ensure_ascii=False, indent=2))


{
  "dangerous_input_blocked": true,
  "path_traversal_blocked": true,
  "retrieval_has_hits": true,
  "live_tool_callable": "环境限制: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))"
}


## 收尾

这份 Notebook 现在只保留案例需要的最小讲解。继续扩展时，优先做三件事：

1. 把 JSON 文本协议升级为原生 tool calling。
2. 给本地检索加 embedding / rerank。
3. 把 trace 和评估结果接到结构化日志系统。